# Permute predictions by randomly swapping predictions

- For alternative hypothesis: 2T - 3Q > 0
    - Text adds more info to raw sum predictions than questionnaire responses


## Set Up

In [1]:
import os, sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
from src.perm_res import expand_alpha_rows
import pandas as pd
import numpy as np
from shutil import rmtree
from src.perm_res import get_avg_r2

PERM_LOOP_DIR = BASE_PATH / "perm_res/preds"
perm_map_export_dir = BASE_PATH / "perm_res/switch/map"
perm_base_export_dir = BASE_PATH / "perm_res/switch/base_dfs"
N_PERMS = 1000

## Construct reference df for permutations

- Each column pertains to one of n permutations
- Each row pertains to a single participant
- Values will be 1/-1

In [2]:
# Use arbitrary base data to get DAWBA_ID cols for parent & self
parent_ids = pd.read_parquet(
    BASE_PATH / "data/w_probs/Llama-3.3-70B-Instruct/pt_noshort/parent.parquet"
)["DAWBA_ID"]
self_ids = pd.read_parquet(
    BASE_PATH / "data/w_probs/Llama-3.3-70B-Instruct/pt_noshort/self.parquet"
)["DAWBA_ID"]

In [3]:
def get_perm_df(id_list, loop_dir):
    df = id_list.reset_index(drop=True)
    rand_sequences = []
    for perm_dir in loop_dir.iterdir():
        perm_name = perm_dir.name
        if perm_name == "baseline":
            continue
        with open(perm_dir / "seed.txt") as f:
            seed_for_perm = int(f.read())
        rng = np.random.default_rng(seed_for_perm)
        rand_seq = rng.choice([0, 1], size=len(df))
        rand_sequences.append(pd.Series(rand_seq, name=perm_name))

    return pd.concat([df] + rand_sequences, axis=1)

In [4]:
parent_perm_map = get_perm_df(parent_ids, PERM_LOOP_DIR)
self_perm_map = get_perm_df(self_ids, PERM_LOOP_DIR)

In [5]:
if perm_map_export_dir.exists():
    rmtree(perm_map_export_dir)
perm_map_export_dir.mkdir(exist_ok=False, parents=True)
parent_perm_map.to_parquet(perm_map_export_dir / "parent.parquet")
self_perm_map.to_parquet(perm_map_export_dir / "self.parquet")

## Get df containing diffs for each participant/model/repeat

In [ ]:
def merge_2_3_dfs(impact_type):
    """
    Merge predictions from 2T and 3Q
        1) Get predictions from best 3Q model (measured by r^2)
        2) Match those predictions with those of all 2T models by DAWBA_ID/repeat

    Single row pertains to a unique participant/repeat/2T_model_config combination
    """
    og_df = pd.read_parquet(
        BASE_PATH
        / f"perm_res/preds/baseline/{impact_type}/pred_df_combined_new.parquet"
    )
    keep_cols = [
        "DAWBA_ID",
        "true",
        "repeat",
        "agg_type",
        "llama_model",
        "desc_seg",
        "alpha",
    ]
    og_df_2_layers = og_df[og_df["num_layers"] == 2][keep_cols + ["pred_stage3_total"]]
    og_df_2_layers = og_df_2_layers.rename(columns={"pred_stage3_total": "pred_2T"})
    og_df_3_layers = og_df[og_df["num_layers"] == 3]
    r2_df = get_avg_r2(og_df_3_layers)
    unique_cols = [
        "DAWBA_ID",
        "true",
        "repeat",
        "pred_stage2_total",
    ]
    best_alpha = (
        r2_df.sort_values("r2_Q", ascending=False).iloc[0]["alpha"].split("_")[0]
    )
    og_df_3_layers_sub = og_df_3_layers[
        og_df_3_layers["alpha"].str.split("_").str[0] == best_alpha
    ][unique_cols]
    og_df_3_layers_sub["alpha"] = best_alpha
    og_df_3_layers_sub = og_df_3_layers_sub.drop_duplicates()
    og_df_3_layers_sub = og_df_3_layers_sub.rename(
        columns={"pred_stage2_total": "pred_3Q"}
    )
    merged = og_df_2_layers.merge(
        og_df_3_layers_sub,
        on=["DAWBA_ID", "true", "repeat"],
        how="left",
        suffixes=("_2T", "_3Q"),
    )
    # Ensure expected length
    unique_model_configs = len(
        merged.drop_duplicates(
            [
                "agg_type",
                "llama_model",
                "desc_seg",
                "alpha_2T",
            ]
        )
    )
    num_repeats = 10
    unique_ids = merged["DAWBA_ID"].nunique()
    assert unique_model_configs * num_repeats * unique_ids == len(merged)
    return merged

In [9]:
parent_base_df = merge_2_3_dfs("parent")
self_base_df = merge_2_3_dfs("self")

In [10]:
if perm_base_export_dir.exists():
    rmtree(perm_base_export_dir)
perm_base_export_dir.mkdir(exist_ok=False, parents=True)
parent_base_df.to_parquet(perm_base_export_dir / "parent.parquet")
self_base_df.to_parquet(perm_base_export_dir / "self.parquet")

## Permute (switch) base_df diffs based on perm_maps

In [11]:
parent_perm_map = pd.read_parquet(perm_map_export_dir / "parent.parquet")
self_perm_map = pd.read_parquet(perm_map_export_dir / "self.parquet")

parent_base_df = pd.read_parquet(perm_base_export_dir / "parent.parquet")
self_base_df = pd.read_parquet(perm_base_export_dir / "self.parquet")

In [12]:
def perm_by_switching(base_df, map_df):
    """
    Switch instead of multiplying diffs by 0 so that we can extract r^2 later
    """
    base_map_merge = base_df.merge(map_df, on="DAWBA_ID")
    cols_to_drop = []
    new_cols = {}
    for perm_col in map_df.columns:
        if perm_col == "DAWBA_ID":
            continue
        mask = base_map_merge[perm_col]
        new_cols[f"pred_2T_{perm_col}"] = np.where(
            mask, base_map_merge["pred_3Q"], base_map_merge["pred_2T"]
        )
        new_cols[f"pred_3Q_{perm_col}"] = np.where(
            mask, base_map_merge["pred_2T"], base_map_merge["pred_3Q"]
        )
        cols_to_drop.append(perm_col)
    base_map_merge = pd.concat(
        [
            base_map_merge.drop(columns=cols_to_drop),
            pd.DataFrame(new_cols, index=base_map_merge.index),
        ],
        axis=1,
    )
    return base_map_merge

In [13]:
parent_permed = perm_by_switching(parent_base_df, parent_perm_map)
self_permed = perm_by_switching(self_base_df, self_perm_map)

In [14]:
permed_df_export_dir = BASE_PATH / "perm_res/switch/perm_diffs"
if permed_df_export_dir.exists():
    rmtree(permed_df_export_dir)
permed_df_export_dir.mkdir(exist_ok=False, parents=True)
parent_permed.to_parquet(permed_df_export_dir / "parent.parquet")
self_permed.to_parquet(permed_df_export_dir / "self.parquet")